# BI Final Project — Online Retail Customer Segmentation

**Course:** Business Intelligence  
**Project topic:** Customer segmentation for targeted marketing  
**Dataset:** Online Retail transactions  

This notebook follows the BI pipeline required for the final project:

1. Business framing and KPIs  
2. Data preparation and EDA  
3. Model and evaluation  
4. Communication and recommendation  
5. Ethics and limitations  

The goal is not to make the most complex model. The goal is to create a simple, understandable analysis that a marketing team could actually use.

## STAGE 1 — Frame & KPIs

### Business question

How can an online retail company segment its customers based on purchasing behavior using **Recency, Frequency, and Monetary value (RFM)** in order to identify high-value, at-risk, and low-engagement customer groups for targeted marketing?

### Decision-maker

The decision-maker is the **Marketing Manager / E-commerce Manager**.

### Business decision

The marketing team will decide which customer segments should receive:

- VIP loyalty campaigns
- Reactivation or win-back campaigns
- Regular promotional campaigns

### Main KPIs

We will use the following KPIs:

- **Recency:** how many days since the customer's last purchase
- **Frequency:** how many purchases the customer made
- **Monetary:** how much money the customer spent
- **Number of customers by segment**
- **Revenue by segment**

### Type of model

This is an **unsupervised learning** problem because the dataset does not have a target column such as `Churn`, `HighValue`, or `Segment`. We will use **K-Means clustering** to discover customer groups.

## Setup

The code is written to be simple and reproducible.  
Keep this notebook in the same folder as either:

- `online+retail.zip`, or
- `Online Retail.xlsx`

Then run the notebook from top to bottom.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42

## Load the dataset

This cell loads the Excel file. If the Excel file is inside the ZIP file, the notebook extracts it automatically.

In [ ]:
zip_path = Path("online+retail.zip")
excel_path = Path("Online Retail.xlsx")

if excel_path.exists():
    data_file = excel_path
elif zip_path.exists():
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("data")
    data_file = Path("data") / "Online Retail.xlsx"
else:
    raise FileNotFoundError("Please upload online+retail.zip or Online Retail.xlsx to the same folder as this notebook.")

print("Using file:", data_file)

df = pd.read_excel(data_file)
df.head()

## STAGE 2 — Prepare data and understand the dataset

Before modeling, we first check the size, columns, missing values, and basic statistics.

In [ ]:
print("Rows and columns:", df.shape)
print()
print("Column names:")
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
missing_values = df.isna().sum()
missing_values

In [ ]:
df.describe()

### Initial observations

Each row represents one product line inside an invoice. This means one invoice can appear in several rows if the customer bought different products.

Important columns for our project:

- `InvoiceNo`: invoice number
- `Quantity`: number of units purchased
- `InvoiceDate`: purchase date
- `UnitPrice`: price per unit
- `CustomerID`: customer identifier
- `Country`: customer country

For customer segmentation, we need customer-level data. That is why we will aggregate transactions into RFM variables.

### Data cleaning

We apply simple cleaning rules:

1. Remove rows with missing `CustomerID`, because we cannot segment customers without an ID.
2. Remove rows with `Quantity <= 0`, because those are usually returns or invalid transactions.
3. Remove rows with `UnitPrice <= 0`, because they do not represent valid revenue.
4. Remove invoice numbers starting with `C`, because those usually represent cancelled invoices.
5. Create a new column called `Revenue`.

In [ ]:
df_clean = df.copy()

original_rows = df_clean.shape[0]

# 1. Keep only rows with CustomerID
df_clean = df_clean.dropna(subset=["CustomerID"])

# 2. Keep only positive quantities
df_clean = df_clean[df_clean["Quantity"] > 0]

# 3. Keep only positive prices
df_clean = df_clean[df_clean["UnitPrice"] > 0]

# 4. Remove cancelled invoices
df_clean = df_clean[~df_clean["InvoiceNo"].astype(str).str.startswith("C")]

# 5. Create revenue
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]

print("Original rows:", original_rows)
print("Clean rows:", df_clean.shape[0])
print("Rows removed:", original_rows - df_clean.shape[0])

df_clean.head()

In [ ]:
print("Date range:", df_clean["InvoiceDate"].min(), "to", df_clean["InvoiceDate"].max())
print("Unique customers:", df_clean["CustomerID"].nunique())
print("Unique invoices:", df_clean["InvoiceNo"].nunique())
print("Unique products:", df_clean["StockCode"].nunique())
print("Total revenue:", round(df_clean["Revenue"].sum(), 2))

### Simple EDA

These charts help us understand the business before modeling.

In [ ]:
country_transactions = df_clean["Country"].value_counts().head(10)

plt.figure(figsize=(8, 4))
country_transactions.plot(kind="bar")
plt.title("Top 10 Countries by Number of Transactions")
plt.xlabel("Country")
plt.ylabel("Transactions")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
country_revenue = df_clean.groupby("Country")["Revenue"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 4))
country_revenue.plot(kind="bar")
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Country")
plt.ylabel("Revenue")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
monthly_revenue = df_clean.set_index("InvoiceDate").resample("M")["Revenue"].sum()

plt.figure(figsize=(9, 4))
monthly_revenue.plot(kind="line", marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

## Create RFM features

RFM is a common way to describe customer purchasing behavior:

- **Recency:** days since the last purchase. Lower is better.
- **Frequency:** number of purchases. Higher is better.
- **Monetary:** total money spent. Higher is better.

We calculate RFM at the customer level.

In [ ]:
reference_date = df_clean["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df_clean.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum")
).reset_index()

rfm.head()

In [ ]:
rfm.describe()

### Why we transform the RFM values

Some customers spend much more than others. If we use the raw numbers, a few extreme values can dominate the model.

To keep the model simple but fairer, we use `log1p`. This reduces very large values while keeping the same order: customers who spend more still have higher values.

Then we scale the values because K-Means uses distance, and distance-based models work better when variables are on a similar scale.

In [ ]:
rfm_features = rfm[["Recency", "Frequency", "Monetary"]].copy()

# Reduce the effect of very large values
rfm_log = np.log1p(rfm_features)

# Scale the data
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

print("Scaled data shape:", rfm_scaled.shape)

## STAGE 3 — Model and evaluation

We use **K-Means clustering** to segment customers.

We test different values of `k` using:

- **Inertia:** lower means clusters are more compact.
- **Silhouette score:** higher means clusters are more separated.

However, this is a business project. We also need the clusters to be easy to explain and useful for marketing.

In [ ]:
k_values = range(2, 8)
inertia_values = []
silhouette_values = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    cluster_labels = model.fit_predict(rfm_scaled)
    
    inertia_values.append(model.inertia_)
    silhouette_values.append(silhouette_score(rfm_scaled, cluster_labels))

k_results = pd.DataFrame({
    "k": list(k_values),
    "inertia": inertia_values,
    "silhouette": silhouette_values
})

k_results

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(k_results["k"], k_results["inertia"], marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(k_results["k"], k_results["silhouette"], marker="o")
plt.title("Silhouette Score by k")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.tight_layout()
plt.show()

### Final choice: k = 3

We choose **k = 3** because it matches the business goal of creating three clear marketing groups:

1. High-value customers
2. At-risk / low-engagement customers
3. Regular customers with growth potential

This choice is also easier to communicate to a non-technical stakeholder. More clusters could give more detail, but they would be harder to explain in a 15-minute presentation.

In [ ]:
FINAL_K = 3

kmeans = KMeans(n_clusters=FINAL_K, random_state=RANDOM_STATE, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

rfm["Cluster"].value_counts().sort_index()

In [ ]:
cluster_summary = rfm.groupby("Cluster").agg(
    Customers=("CustomerID", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Total_Revenue=("Monetary", "sum")
).round(2)

cluster_summary

### Name the clusters

K-Means only gives numbers like 0, 1, and 2. These numbers do not mean anything by themselves.

We must translate them into business names by looking at their average Recency, Frequency, and Monetary values.

In [ ]:
# Find the cluster with the highest average spending
high_value_cluster = cluster_summary["Avg_Monetary"].idxmax()

# Find the cluster with the highest recency, meaning customers have not bought recently
low_engagement_cluster = cluster_summary["Avg_Recency"].idxmax()

segment_names = {}

for cluster in cluster_summary.index:
    if cluster == high_value_cluster:
        segment_names[cluster] = "High-Value Customers"
    elif cluster == low_engagement_cluster:
        segment_names[cluster] = "At-Risk / Low-Engagement Customers"
    else:
        segment_names[cluster] = "Regular / Growth Potential Customers"

segment_names

In [ ]:
rfm["Segment"] = rfm["Cluster"].map(segment_names)

segment_summary = rfm.groupby("Segment").agg(
    Customers=("CustomerID", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Total_Revenue=("Monetary", "sum")
).round(2).sort_values("Total_Revenue", ascending=False)

segment_summary

### Segment interpretation

The final segment names are based on the data:

- **High-Value Customers:** customers with the strongest monetary value and purchase activity.
- **At-Risk / Low-Engagement Customers:** customers with high recency, meaning they have not purchased recently.
- **Regular / Growth Potential Customers:** customers in the middle. They are not the highest-value group, but they still have potential for marketing actions.

## STAGE 4 — Communicate results and recommendations

This section creates simple dashboard-style charts and a recommendation table for the marketing team.

In [ ]:
plt.figure(figsize=(8, 4))
segment_summary["Customers"].plot(kind="bar")
plt.title("Number of Customers by Segment")
plt.xlabel("Segment")
plt.ylabel("Number of customers")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
segment_summary["Total_Revenue"].plot(kind="bar")
plt.title("Total Revenue by Segment")
plt.xlabel("Segment")
plt.ylabel("Total revenue")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
segment_summary["Avg_Recency"].plot(kind="bar")
plt.title("Average Recency by Segment")
plt.xlabel("Segment")
plt.ylabel("Average recency in days")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
segment_summary["Avg_Frequency"].plot(kind="bar")
plt.title("Average Frequency by Segment")
plt.xlabel("Segment")
plt.ylabel("Average number of purchases")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
recommendations = pd.DataFrame({
    "Segment": [
        "High-Value Customers",
        "At-Risk / Low-Engagement Customers",
        "Regular / Growth Potential Customers"
    ],
    "Business meaning": [
        "Best customers. They spend more and are valuable for the company.",
        "Customers who have not purchased recently and may be lost or inactive.",
        "Customers with moderate activity who could buy more with the right campaign."
    ],
    "Recommended action": [
        "Offer VIP benefits, loyalty rewards, and early access to products.",
        "Use low-cost win-back emails or reactivation discounts.",
        "Use product recommendations, bundles, and cross-selling campaigns."
    ],
    "KPI to monitor": [
        "Retention rate and revenue from VIP customers",
        "Return purchase rate after win-back campaign",
        "Increase in frequency and average monetary value"
    ]
})

recommendations

### Main recommendation

The marketing team should not send the same campaign to every customer. Instead, it should use the customer segments to design different actions:

- Protect high-value customers with loyalty benefits.
- Reactivate at-risk / low-engagement customers using low-cost campaigns.
- Grow regular customers through product recommendations and bundles.

This makes the marketing strategy more targeted and more connected to customer behavior.

### Optional: save outputs for the presentation and dashboard

These files can be included in the final ZIP or used to build the presentation.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# Save CSV outputs
rfm.to_csv(output_dir / "customer_segments_output.csv", index=False)
segment_summary.to_csv(output_dir / "segment_summary_output.csv")
recommendations.to_csv(output_dir / "marketing_recommendations.csv", index=False)

print("Files saved in the outputs folder.")

In [ ]:
# Save simple dashboard charts as images

plt.figure(figsize=(8, 4))
segment_summary["Customers"].plot(kind="bar")
plt.title("Number of Customers by Segment")
plt.xlabel("Segment")
plt.ylabel("Number of customers")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(output_dir / "customers_by_segment.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 4))
segment_summary["Total_Revenue"].plot(kind="bar")
plt.title("Total Revenue by Segment")
plt.xlabel("Segment")
plt.ylabel("Total revenue")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(output_dir / "revenue_by_segment.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 4))
segment_summary["Avg_Recency"].plot(kind="bar")
plt.title("Average Recency by Segment")
plt.xlabel("Segment")
plt.ylabel("Average recency in days")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(output_dir / "recency_by_segment.png", dpi=150)
plt.show()

In [ ]:
html = f'''
<html>
<head>
    <title>Online Retail Customer Segmentation Dashboard</title>
</head>
<body>
    <h1>Online Retail Customer Segmentation Dashboard</h1>
    <p>This simple dashboard summarizes the customer segments created with K-Means clustering.</p>

    <h2>Segment Summary</h2>
    {segment_summary.to_html()}

    <h2>Marketing Recommendations</h2>
    {recommendations.to_html(index=False)}

    <h2>Charts</h2>
    <h3>Number of Customers by Segment</h3>
    <img src="outputs/customers_by_segment.png" width="700">

    <h3>Total Revenue by Segment</h3>
    <img src="outputs/revenue_by_segment.png" width="700">

    <h3>Average Recency by Segment</h3>
    <img src="outputs/recency_by_segment.png" width="700">
</body>
</html>
'''

with open("online_retail_segmentation_dashboard.html", "w", encoding="utf-8") as f:
    f.write(html)

print("HTML dashboard saved as online_retail_segmentation_dashboard.html")

## STAGE 5 — Ethics, bias, and limitations

### Privacy

The dataset uses `CustomerID`, but it does not include names, emails, addresses, or phone numbers. Even so, customer IDs should still be treated carefully because they represent individual customers.

### Bias and fairness

The data is mostly from one retail context and may be concentrated in specific countries. Because of this, the segments may not represent all types of customers or all markets equally.

### Limitations

- This is a segmentation model, not a prediction model.
- The model does not prove why customers behave in a certain way.
- The data is historical, so customer behavior may have changed.
- K-Means depends on the variables we choose.
- The cluster names are business interpretations, not automatic truths.
- We removed returns and invalid transactions, which makes the analysis cleaner but may hide return behavior.

### Responsible use

The segments should support marketing decisions, but they should not be used to unfairly exclude customers. The company should test campaigns carefully and monitor results after using the segmentation.

## Final conclusion

This project used Online Retail transaction data to create customer segments based on RFM behavior.

The main result is a simple segmentation with three groups:

1. **High-Value Customers**
2. **At-Risk / Low-Engagement Customers**
3. **Regular / Growth Potential Customers**

The recommendation is to use different marketing actions for each group instead of sending the same campaign to all customers.

This approach gives the marketing team a clear and actionable way to improve targeting, retention, and revenue.

## AI assistance disclosure

AI tools were used as support to organize the notebook, improve explanations, and help structure the code. The team reviewed the notebook and must be able to explain every line of code and every business decision in the presentation.

## Reproducibility checklist

- Random seed is set with `RANDOM_STATE = 42`.
- The notebook does not use absolute local paths.
- The dataset is loaded from the same folder as the notebook.
- The notebook creates output files for the dashboard and presentation.
- The notebook should run from top to bottom using **Restart & Run All**.